# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("SessaoLocal")
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_tim = Path("/assets/data/cdr/teleparser/Semana91/tim/volte")
_files_tim = [str(file) for file in folder_tim.glob("*.parquet")]
df_tim = pd.DataFrame(_files_tim, columns=["file_path"])
df_tim["exchange_id"] = df_tim["file_path"].str.extract(r"\/([A-Z0-9_]+)\.")
files_tim = df_tim.groupby("exchange_id").sample(4,random_state=22)["file_path"].to_list()

### Arquivos intermediário e final

In [ ]:
extracted_file = "/assets/data/cdr/teletools/db/extraidos/tim_volte_extraido.parquet"
transformed_file = "/assets/data/cdr/teletools/db/transformados/tim_volte_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
df = spark.read.parquet(*files_tim)
df.groupBy("recordType").count().orderBy("recordType").show(truncate=False)

## Extração

In [ ]:
%%time
extractor = CDRParquetExtractor(spark)
extractor.extract_cdr(files_tim, extracted_file, "smp_huawei_volte_tim")

In [ ]:
df_extracted = spark.read.parquet(extracted_file)
df_extracted.show(5)

## Transformação

### Número de origem

In [ ]:
# Define a janela agrupada por tipo_cdr com ordenação aleatória
window_spec = Window.partitionBy("tipo_cdr").orderBy(F.rand(seed=42))

columns_to_keep = ["tipo_cdr", "_numero_origem_auth", "_numero_origem_ats_ibcf"]

# Atribui um número de linha aleatório por grupo e filtra as 10 primeiras
df_origem = (
    df_extracted.select(columns_to_keep).withColumn("row_num", F.row_number().over(window_spec))
      .filter(F.col("row_num") <= 10)
      .drop("row_num")
)

df_origem.show(truncate=False)

In [ ]:
is_ats = F.col("tipo_cdr") == "aTSRecord"
is_ibcf = F.col("tipo_cdr") == "iBCFRecord"

sip_extract_pattern = r"sip:([0-9]+)[@;]"

# Números de origem ATS estão com os dígitos invertidos 2 a 2.
# Por exemplo, o número "11551899781480F2" seria invertido para "115581998741082F".
# 11-55-18-99-78-14-80-F2
#  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓
# 11-55-81-99-87-41-08-2F
# Além disso trazem prefixo 11 ou 14 que devem ser removidos para não serem confundidos como os respectivos CN
extract_ats_calling_party = F.get_json_object(F.col("_numero_origem_ats_ibcf"), "$[0].tEL-URI")
ats_calling_party = F.regexp_replace(
    F.get_json_object(F.col("_numero_origem_ats_ibcf"), "$[0].tEL-URI").substr(3, 9999),
    "(.)(.)",
    "$2$1",
)

ibfc_calling_party = F.regexp_extract(
    F.get_json_object(F.col("_numero_origem_ats_ibcf"), "$[0].sIP-URI"),
    sip_extract_pattern,
    1,
)

df_origem = df_origem.withColumn(
    "numero_origem",
    F.when(is_ats, ats_calling_party).when(is_ibcf, ibfc_calling_party)
).withColumn(
    "_numero_origem_original",
    F.when(is_ats, extract_ats_calling_party).otherwise(F.col("numero_origem"))
)

df_origem.show(truncate=False)

### Número de destino

In [ ]:
columns_to_keep = ["tipo_cdr", "_numero_destino_ats", "_numero_destino_ibcf"]
# Atribui um número de linha aleatório por grupo e filtra as 10 primeiras
df_destino = (
    df_extracted.select(columns_to_keep).withColumn("row_num", F.row_number().over(window_spec))
      .filter(F.col("row_num") <= 10)
      .drop("row_num")
)

df_destino.show(truncate=False)

In [ ]:
ats_called_party = F.regexp_replace(
    F.col("_numero_destino_ats").substr(3, 9999),
    "(.)(.)",
    "$2$1",
)

ibfc_called_party = F.regexp_extract(
    F.col("_numero_destino_ibcf"),
    sip_extract_pattern,
    1,
)

df_destino = df_destino.withColumn(
    "numero_destino",
    F.when(is_ats, ats_called_party)
        .when(is_ibcf, ibfc_called_party)
).withColumn("_numero_destino_original", F.when(is_ats, F.col("_numero_destino_ats")).otherwise(F.col("numero_destino")))

df_destino.show(truncate=False)

### xCGI

In [ ]:
_CELL_EXTRACT_PATTERN = r"3gpp=([0-9a-fA-F]+);?"

def _extract_cell_info(
    df,
    col_name,
    out_col_tec = "_tecnologia_celula",
    out_col_cell_id = "_id_celula_hex",
):
    """Extrai informações de célula a partir da coluna de rede.

    Args:
        df: DataFrame de entrada.
        col_name: Nome da coluna contendo a informação de rede.
        out_col_tec: Nome da coluna de saída para a tecnologia da célula, ou
            ``None`` para não gerar essa coluna.
        out_col_cell_id: Nome da coluna de saída para o id hexadecimal da
            célula, ou ``None`` para não gerar essa coluna.

    Returns:
        DataFrame: Cópia do DataFrame de entrada com as colunas
        as colunas especificadas em ``out_col_tec`` e ``out_col_cell_id`` adicionadas.

    Raises:
        ValueError: Se ``df`` ou ``col_name`` não forem informados, ou se
            ``col_name`` não existir entre as colunas de ``df``.
    """
    if df is None:
        raise ValueError("O parâmetro 'df' é obrigatório e não foi informado.")
    if not col_name:
        raise ValueError("O parâmetro 'col_name' é obrigatório e não foi informado.")
    if col_name not in df.columns:
        raise ValueError(
            f"A coluna de origem '{col_name}' não existe no DataFrame. "
            f"Colunas disponíveis: {df.columns}"
        )

    network_info = F.col(col_name)
    new_columns = {}
    if out_col_tec is not None:
        new_columns[out_col_tec] = F.split(network_info, ";").getItem(0)
    if out_col_cell_id is not None:
        new_columns[out_col_cell_id] = F.regexp_extract(
            network_info, _CELL_EXTRACT_PATTERN, 1
        )

    return df.withColumns(new_columns)

In [ ]:
def _format_cell_id(df, col_name, out_col, gnb_id_bits=26, output_format="default"):
    col = F.col(col_name)
    length = F.length(col)

    canonical_out_col = f"{out_col}_canonical"

    # ---- 3G (UTRAN, 13 chars) ----
    tac_3g = F.conv(F.substring(col, 6, 4), 16, 10).cast("long")
    ci_3g = F.conv(F.substring(col, 10, 4), 16, 10).cast("long")

    ci_formatted = F.concat_ws(
        "-",
        F.substring(col, 1, 3),  # mcc
        F.substring(col, 4, 2),  # mnc
        F.lpad(tac_3g.cast("string"), 5, "0"),  # tac (16 bits)
        F.lpad(ci_3g.cast("string"), 5, "0"),  # ci (16 bits)
    )
    ci_canonical = F.concat_ws(
        "-",
        F.substring(col, 1, 3),  # mcc
        F.substring(col, 4, 2),  # mnc
        F.concat_ws("", tac_3g.cast("string"), ci_3g.cast("string")),  # ci (16 bits)
    )

    # ---- 4G (ECGI, 16 chars) ----
    ecgi_val = F.conv(F.substring(col, 10, 7), 16, 10).cast("long")
    ecgi_formatted = F.concat_ws(
        "-",
        F.substring(col, 1, 3),  # mcc
        F.substring(col, 4, 2),  # mnc
        F.lpad(
            (ecgi_val / 256).cast("long").cast("string"), 7, "0"
        ),  # enb_id (20 bits)
        F.lpad((ecgi_val % 256).cast("string"), 3, "0"),  # cell_id (8 bits)
    )
    ecgi_canonical = F.concat_ws(
        "-",
        F.substring(col, 1, 3),  # mcc
        F.substring(col, 4, 2),  # mnc
        F.concat_ws(
            "",
            (ecgi_val / 256).cast("long").cast("string"),
            (ecgi_val % 256).cast("string")
        )
    )

    # ---- 5G (NCGI, 20 chars) ----
    # NCGI = 36 bits totais. gNB ID = 26 bits (default), Cell ID = 36 - 26 = 10 bits
    cell_id_bits = 36 - gnb_id_bits  # 10
    cell_id_mask = (1 << cell_id_bits) - 1  # 0x3FF = 1023
    ncgi_val = F.conv(F.substring(col, 12, 9), 16, 10).cast("long")

    ncgi_formatted = F.concat_ws(
        "-",
        F.substring(col, 1, 3),  # mcc
        F.substring(col, 4, 2),  # mnc
        F.lpad(
            F.shiftright(ncgi_val, cell_id_bits).cast("string"), 8, "0"
        ),  # gnb_id (26 bits)
        F.lpad(
            ncgi_val.bitwiseAND(cell_id_mask).cast("string"), 4, "0"
        ),  # cell_id (10 bits)
    )
    ncgi_canonical = F.concat_ws(
        "-",
        F.substring(col, 1, 3),  # mcc
        F.substring(col, 4, 2),  # mnc
        F.concat_ws(
            "",
            F.shiftright(ncgi_val, cell_id_bits).cast("string"),
            ncgi_val.bitwiseAND(cell_id_mask).cast("string")
        )
    )

    formatted_cell_id = (
        F.when(length == 13, ci_formatted)
        .when(length == 16, ecgi_formatted)
        .when(length == 20, ncgi_formatted)
        .otherwise(col)
    )
    canonical_cell_id = (
        F.when(length == 13, ci_canonical)
        .when(length == 16, ecgi_canonical)
        .when(length == 20, ncgi_canonical)
        .otherwise(col)
    )

    return df.withColumn(
        out_col,
        F.when(formatted_cell_id == "", F.lit(None)).otherwise(formatted_cell_id),
    ).withColumn(
        canonical_out_col, 
        F.when(formatted_cell_id == "", F.lit(None)).otherwise(canonical_cell_id),
    )

In [ ]:
df_rede = _extract_cell_info(df_extracted.select("_informacao_rede"), "_informacao_rede")
df_rede.show(5,truncate=False)

In [ ]:
df_rede = _format_cell_id(df_rede, "_id_celula_hex", "_id_celula")
df_rede.show(5,truncate=False)

### Transformer

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_smp_huawei_volte_tim(extracted_file, transformed_file)

In [ ]:
df_transformed = spark.read.parquet(transformed_file)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df = df_transformed.filter(F.col("nu_cgi_destino_hex").isNotNull()).select("nu_cgi_destino_hex","no_tecnologia_celula_destino", "nu_cgi_destino").limit(10)
df = _format_cell_id(df, "nu_cgi_destino_hex", "_id_celula")
df.show()

In [ ]:
len("7240379D8C11480A")